# Model Monitoring & CloudWatch Dashboard
This notebook operationalizes the project monitoring requirements. We will implement **Data Quality**, **Model Quality**, and **Infrastructure** monitors, and consolidate them into a unified **CloudWatch Dashboard**.

In [1]:
!pip install "sagemaker==2.214.0"

In [2]:
import os
import boto3
import sagemaker
import pandas as pd
import time
from datetime import datetime, timedelta
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    EndpointInput,
    DatasetFormat
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name
prefix = "nyc-collisions-monitoring"

print(f"Monitoring initialized in {region} for bucket {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Monitoring initialized in us-east-1 for bucket sagemaker-us-east-1-594126158441


## 1. Deploy Real-Time Endpoint
To use SageMaker Model Monitor, we need a live endpoint with **Data Capture** enabled. This captures all inference requests and saves them to S3 for analysis.

In [3]:
#  Define S3 Capture Path
data_capture_uri = f"s3://{bucket}/{prefix}/datacapture"

# Dynamically fetch the latest model from the project prefix
sm_client = boto3.client("sagemaker")
training_jobs = sm_client.list_training_jobs(
    NameContains="sagemaker-scikit-learn",
    StatusEquals="Completed",
    SortBy="CreationTime",
    SortOrder="Descending"
)

if training_jobs["TrainingJobSummaries"]:
    latest_job_name = training_jobs["TrainingJobSummaries"][0]["TrainingJobName"]
    model_artifact = sm_client.describe_training_job(TrainingJobName=latest_job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"Found latest model: {model_artifact}")
else:
    raise ValueError("No completed training jobs found to deploy.")

# Create Model Entity
image_uri = sagemaker.image_uris.retrieve(framework="sklearn", version="1.2-1", region=region)
model = Model(
    image_uri=image_uri, 
    model_data=model_artifact, 
    role=role,
    entry_point='sagemaker_train.py',
    source_dir='../src'
)

# Setup Data Capture
capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=data_capture_uri
)

# Deploy
endpoint_name = f"collisions-monitor-ep-{int(time.time())}"
model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=capture_config
)

predictor = Predictor(endpoint_name=endpoint_name, serializer=CSVSerializer())
print(f"Endpoint {endpoint_name} is LIVE with Data Capture.")

Found latest model: s3://sagemaker-us-east-1-594126158441/sagemaker-scikit-learn-2026-06-09-04-12-15-570/output/model.tar.gz


-

-

-

-

-

-

!

Endpoint collisions-monitor-ep-1781372955 is LIVE with Data Capture.


## 2. Data Quality Monitoring
We will baseline the validation data to identify **Data Drift** (e.g., if Borough distributions change).

In [4]:
data_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

# Use the validation split from Notebook 04
training_prefix = "aai-540-group6/nyc-collisions-ml"
baseline_data_uri = f"s3://{bucket}/{training_prefix}/validation/val.csv"
print(f"Using baseline data: {baseline_data_uri}")

data_monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/data_quality_results",
    wait=True
)

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-13-17-52-52-694


Using baseline data: s3://sagemaker-us-east-1-594126158441/aai-540-group6/nyc-collisions-ml/validation/val.csv


.

.

.

...

2026-06-13 17:58:09,373 INFO cluster.YarnSchedulerBackend$YarnDriverEndpoint: Registered executor NettyRpcEndpointRef(spark-client://Executor) (10.0.124.203:60084) with ID 1,  ResourceProfileId 0
2026-06-13 17:58:09,770 INFO storage.BlockManagerMasterEndpoint: Registering block manager algo-1:37535 with 2.7 GiB RAM, BlockManagerId(1, algo-1, 37535, None)


2026-06-13 17:58:13,610 INFO cluster.YarnClientSchedulerBackend: SchedulerBackend is ready for scheduling beginning after waiting maxRegisteredResourcesWaitingTime: 30000000000(ns)
2026-06-13 17:58:14,146 WARN spark.SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/tmp' appears to be on the local filesystem.
2026-06-13 17:58:14,370 INFO internal.SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
2026-06-13 17:58:14,385 INFO internal.SharedState: Warehouse path is 'file:/usr/spark-3.3.0/spark-warehouse'.
2026-06-13 17:58:16,753 INFO datasources.InMemoryFileIndex: It took 87 ms to list leaf files for 1 paths.
2026-06-13 17:58:17,043 INFO memory.MemoryStore: Block broadcast_0 stored as values in memory (estimated size 416.9 KiB, free 1458.2 MiB)
2026-06-13 17:58:17,633 INFO memory.MemoryStore: Block broadcast_0_piece0 stored as bytes in memory (estimated siz

2026-06-13 17:58:23,964 INFO datasources.FileSourceStrategy: Pushed Filters: 
2026-06-13 17:58:23,966 INFO datasources.FileSourceStrategy: Post-Scan Filters: 
2026-06-13 17:58:23,970 INFO datasources.FileSourceStrategy: Output Data Schema: struct<borough: string, month: string, hour: string, is_rush_hour: string, is_weekend: string ... 6 more fields>
2026-06-13 17:58:24,235 INFO memory.MemoryStore: Block broadcast_2 stored as values in memory (estimated size 416.5 KiB, free 1457.7 MiB)
2026-06-13 17:58:24,316 INFO memory.MemoryStore: Block broadcast_2_piece0 stored as bytes in memory (estimated size 39.1 KiB, free 1457.7 MiB)
2026-06-13 17:58:24,316 INFO storage.BlockManagerInfo: Added broadcast_2_piece0 in memory on 10.0.124.203:33787 (size: 39.1 KiB, free: 1458.5 MiB)
2026-06-13 17:58:24,318 INFO spark.SparkContext: Created broadcast 2 from head at DataAnalyzer.scala:124
2026-06-13 17:58:24,356 INFO execution.FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes

2026-06-13 17:58:30,850 INFO scheduler.TaskSetManager: Finished task 0.0 in stage 2.0 (TID 2) in 2440 ms on algo-1 (executor 1) (1/1)
2026-06-13 17:58:30,850 INFO cluster.YarnScheduler: Removed TaskSet 2.0, whose tasks have all completed, from pool 
2026-06-13 17:58:30,858 INFO scheduler.DAGScheduler: ShuffleMapStage 2 (collect at AnalysisRunner.scala:326) finished in 2.504 s
2026-06-13 17:58:30,860 INFO scheduler.DAGScheduler: looking for newly runnable stages
2026-06-13 17:58:30,860 INFO scheduler.DAGScheduler: running: Set()
2026-06-13 17:58:30,861 INFO scheduler.DAGScheduler: waiting: Set()
2026-06-13 17:58:30,861 INFO scheduler.DAGScheduler: failed: Set()
2026-06-13 17:58:30,969 INFO spark.SparkContext: Starting job: collect at AnalysisRunner.scala:326
2026-06-13 17:58:30,972 INFO scheduler.DAGScheduler: Got job 3 (collect at AnalysisRunner.scala:326) with 1 output partitions
2026-06-13 17:58:30,972 INFO scheduler.DAGScheduler: Final stage: ResultStage 4 (collect at AnalysisRunner

2026-06-13 17:58:40,765 INFO memory.MemoryStore: Block broadcast_18 stored as values in memory (estimated size 21.4 KiB, free 1457.1 MiB)
2026-06-13 17:58:40,768 INFO memory.MemoryStore: Block broadcast_18_piece0 stored as bytes in memory (estimated size 10.1 KiB, free 1457.1 MiB)
2026-06-13 17:58:40,773 INFO storage.BlockManagerInfo: Added broadcast_18_piece0 in memory on 10.0.124.203:33787 (size: 10.1 KiB, free: 1458.4 MiB)
2026-06-13 17:58:40,774 INFO spark.SparkContext: Created broadcast 18 from broadcast at DAGScheduler.scala:1513
2026-06-13 17:58:40,775 INFO scheduler.DAGScheduler: Submitting 1 missing tasks from ShuffleMapStage 20 (MapPartitionsRDD[86] at count at StatsGenerator.scala:66) (first 15 tasks are for partitions Vector(0))
2026-06-13 17:58:40,775 INFO cluster.YarnScheduler: Adding task set 20.0 with 1 tasks resource profile 0
2026-06-13 17:58:40,788 INFO scheduler.TaskSetManager: Starting task 0.0 in stage 20.0 (TID 16) (algo-1, executor 1, partition 0, PROCESS_LOCAL,

## 3. Model Quality Monitoring
We will track **Precision, Recall, and Accuracy** by comparing endpoint predictions against ground truth.

In [26]:
import botocore

model_sched_name = f"{endpoint_name}-model-qc"
schedule_already_exists = False
try:
    sm_client.describe_monitoring_schedule(MonitoringScheduleName=model_sched_name)
    schedule_already_exists = True
    print(f"Schedule {model_sched_name} already exists. Nothing to do.")
except botocore.exceptions.ClientError:
    pass

if not schedule_already_exists:
    print("--- Generating predictions for Model Quality baseline ---")
    df_val_local = pd.read_csv("data_splits/val.csv")
    sample_df = df_val_local.sample(250, random_state=42)
    features_only = sample_df.drop("target", axis=1)
    
    preds = []
    for _, row in features_only.iterrows():
        payload = ",".join([str(val) for val in row.values])
        response = predictor.predict(payload)
        clean_resp = response.decode("utf-8").strip().replace("[", "").replace("]", "")
        preds.append(int(float(clean_resp)))
    
    baseline_df = pd.DataFrame({"prediction": preds, "target": sample_df["target"].values})
    
    baseline_local_path = "data_splits/baseline_with_predictions.csv"
    baseline_df.to_csv(baseline_local_path, index=False)
    baseline_s3_uri = sess.upload_data(baseline_local_path, bucket, f"{prefix}/model_quality_baseline_with_preds")
    print(f"Uploaded combined baseline to: {baseline_s3_uri}")

    model_quality_monitor = ModelQualityMonitor(
        max_runtime_in_seconds=1800, # Less than 1 hour (3600s)
        role=role,
        instance_count=1,
        instance_type="ml.m5.large",
        sagemaker_session=sess
    )
    
    model_quality_monitor.suggest_baseline(
        baseline_dataset=baseline_s3_uri,
        dataset_format=sagemaker.model_monitor.dataset_format.DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{bucket}/{prefix}/model_quality_results",
        problem_type="BinaryClassification",
        ground_truth_attribute="target",
        inference_attribute="prediction",
        wait=True
    )

    print(f"\n--- Creating Model Quality Schedule: {model_sched_name} ---")
    model_constraints_s3_path = model_quality_monitor.latest_baselining_job.suggested_constraints().file_s3_uri
    
    model_quality_monitor.create_monitoring_schedule(
        monitor_schedule_name=model_sched_name,
        endpoint_input=EndpointInput(endpoint_name=endpoint_name, destination="/opt/ml/processing/input_data", inference_attribute="0"),
        problem_type="BinaryClassification",
        ground_truth_input=f"s3://{bucket}/{prefix}/ground_truth_data",
        constraints=model_constraints_s3_path,
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True
    )


--- Generating predictions for Model Quality baseline ---


INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-13-19-41-26-074


Uploaded combined baseline to: s3://sagemaker-us-east-1-594126158441/nyc-collisions-monitoring/model_quality_baseline_with_preds/baseline_with_predictions.csv


...

.

.

!


--- Creating Model Quality Schedule: collisions-monitor-ep-1781372955-model-qc ---


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: collisions-monitor-ep-1781372955-model-qc


## 4. CloudWatch Dashboard Implementation
We will now programmatically create a CloudWatch dashboard that tracks:
1. **Infrastructure:** Endpoint CPU & Memory.
2. **Data Quality:** Baseline violations.
3. **Model Quality:** Accuracy & Recall metrics.

In [27]:
dashboard_body = {
    "widgets": [
        {
            "type": "metric", "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [["AWS/SageMaker", "CPUUtilization", "EndpointName", endpoint_name, "VariantName", "AllInstances"]],
                "period": 300, "stat": "Average", "region": region,
                "title": "Infrastructure: CPU Utilization"
            }
        },
        {
            "type": "metric", "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [["AWS/SageMaker", "MemoryUtilization", "EndpointName", endpoint_name, "VariantName", "AllInstances"]],
                "period": 300, "stat": "Average", "region": region,
                "title": "Infrastructure: Memory Utilization"
            }
        },
        {
            "type": "metric", "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["aws/sagemaker/Endpoints/model-metrics", "accuracy", "Endpoint", endpoint_name],
                    [".", "recall", ".", "."]
                ],
                "period": 3600, "stat": "Average", "region": region,
                "title": "Model Quality: Accuracy & Recall"
            }
        },
        {
            "type": "metric", "x": 12, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [["aws/sagemaker/Endpoints/data-metrics", "feature_baseline_drift_total_violations", "Endpoint", endpoint_name]],
                "period": 3600, "stat": "Sum", "region": region,
                "title": "Data Quality: Feature Drift Violations"
            }
        }
    ]
}

cw_client = boto3.client('cloudwatch')
cw_client.put_dashboard(
    DashboardName='NYC-Collision-ML-Performance',
    DashboardBody=json.dumps(dashboard_body)
)

print("CloudWatch Dashboard 'NYC-Collision-ML-Performance' created/updated successfully.")

CloudWatch Dashboard 'NYC-Collision-ML-Performance' created/updated successfully.


In [ ]:
# Cleanup: Uncomment to delete endpoint and avoid costs
# predictor.delete_endpoint()
# predictor.delete_model()